# Day 034 Project: Installable AI CLI Tool

## What You're Building

A multi-command `AICli` instance with at least three subcommands that each transform `--input` text through a different prompt before calling the LLM.

## Project Requirements

1. Create an `AICli` instance stored as `cli`
2. Register at least 3 subcommands (e.g., `ask`, `summarize`, `classify`)
3. Each subcommand must have a distinct `prompt_fn` that wraps the input
4. Call at least 2 different subcommands and print the results
5. Print the entry-point snippet that would make this tool installable
6. Verify with `_run_project_checks()`

## Provided: All Helper Functions + AICli

In [ ]:
from argparse import ArgumentParser

def make_parser() -> ArgumentParser:
    parser = ArgumentParser(
        prog='ai-tool',
        description='AI command-line tool powered by local LLM',
    )
    parser.add_argument(
        '--prompt', '-p', type=str, required=True,
        help='Prompt to send to the model',
    )
    parser.add_argument(
        '--model', '-m', type=str, default='llama3.2',
        help='Ollama model name (default: llama3.2)',
    )
    parser.add_argument(
        '--verbose', '-v', action='store_true',
        help='Print extra diagnostic output',
    )
    return parser


def make_extended_parser() -> ArgumentParser:
    parser = ArgumentParser(
        prog='ai-batch',
        description='AI batch processing tool',
    )
    parser.add_argument(
        '--prompt', '-p', type=str, required=True,
        help='Prompt text',
    )
    parser.add_argument(
        '--count', '-n', type=int, default=1,
        help='Number of completions (default: 1)',
    )
    parser.add_argument(
        '--format', '-f',
        choices=['text', 'json', 'markdown'],
        default='text',
        help='Output format (default: text)',
    )
    parser.add_argument(
        '--temperature', type=float, default=0.7,
        help='Sampling temperature 0.0-1.0 (default: 0.7)',
    )
    return parser


def make_subcommand_parser() -> ArgumentParser:
    parser = ArgumentParser(prog='ai-tool', description='AI CLI')
    subs   = parser.add_subparsers(dest='command', required=True,
                                   title='commands')

    chat = subs.add_parser('chat', help='Send a prompt to the AI')
    chat.add_argument('--prompt', '-p', required=True, help='The prompt')
    chat.add_argument('--model',  '-m', default='llama3.2')

    summarize = subs.add_parser('summarize', help='Summarize text')
    summarize.add_argument('--text',  '-t', required=True,
                           help='Text to summarize')
    summarize.add_argument('--model', '-m', default='llama3.2')

    return parser


def dispatch(ns, handlers: dict) -> str:
    cmd = ns.command
    if cmd not in handlers:
        raise KeyError(f"No handler registered for command: {cmd!r}")
    return handlers[cmd](ns)


import ollama
from argparse import ArgumentParser

class AICli:
    def __init__(self, prog: str = 'ai-tool', model: str = 'llama3.2'):
        self.model       = model
        self._parser     = ArgumentParser(prog=prog,
                                          description='AI command-line tool')
        self._subs       = self._parser.add_subparsers(dest='command',
                                                        required=True)
        self._prompt_fns: dict = {}

    def add_command(self, name: str, prompt_fn,
                    help: str = '') -> 'AICli':
        sub = self._subs.add_parser(name, help=help)
        sub.add_argument('--input', '-i', required=True, help='Input text')
        self._prompt_fns[name] = prompt_fn
        return self

    def run(self, args_list: list[str]) -> str:
        ns     = self._parser.parse_args(args_list)
        prompt = self._prompt_fns[ns.command](ns.input)
        resp   = ollama.chat(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return resp['message']['content']

## Your CLI Tool

In [ ]:
# Build your AICli with at least 3 subcommands
cli = (
    AICli(prog='my-ai', model='llama3.2')
    .add_command(
        'ask',
        lambda t: t,
        help='Send a raw prompt to the AI',
    )
    .add_command(
        'summarize',
        lambda t: f'Summarize in 2 sentences:\n\n{t}',
        help='Summarize the input text',
    )
    .add_command(
        'classify',
        lambda t: f"Classify as positive/negative/neutral. One word.\n\n'{t}'",
        help='Classify text sentiment',
    )
)

# TODO: run at least 2 subcommands and print results
# result1 = cli.run(['ask', '--input', 'What is the capital of France?'])
# print(f'ask: {result1.strip()}')
#
# result2 = cli.run(['classify', '--input', 'I love this!'])
# print(f'classify: {result2.strip()}')

# TODO: print the pyproject.toml entry-point snippet
print('[project.scripts]')
print('my-ai = "my_module:main"')

## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: cli is an AICli instance
    try:
        assert 'cli' in globals()
        assert isinstance(cli, AICli), \
            f'cli should be AICli, got {type(cli)}'
        passed += 1; print('\u2705 Check 1: cli is an AICli instance')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: at least 3 subcommands registered
    try:
        assert hasattr(cli, '_prompt_fns')
        assert len(cli._prompt_fns) >= 3, \
            f'need >= 3 commands, got {len(cli._prompt_fns)}: {list(cli._prompt_fns)}'
        passed += 1; print(f'\u2705 Check 2: {len(cli._prompt_fns)} subcommands registered')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: result1 defined (at least one subcommand was run)
    try:
        assert 'result1' in globals(), \
            'result1 not defined — run at least 2 subcommands'
        assert isinstance(result1, str) and result1.strip(), \
            'result1 should be a non-empty string'
        passed += 1; print('\u2705 Check 3: result1 is a non-empty string')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: result2 defined (at least two subcommands were run)
    try:
        assert 'result2' in globals(), \
            'result2 not defined — run at least 2 subcommands'
        assert isinstance(result2, str) and result2.strip(), \
            'result2 should be a non-empty string'
        passed += 1; print('\u2705 Check 4: result2 is a non-empty string')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: distinct prompt_fns (each command does something different)
    try:
        fns = list(cli._prompt_fns.values())
        prompts = [fn('test input') for fn in fns]
        unique = len(set(prompts))
        assert unique == len(fns), \
            f'all prompt_fns should produce different prompts for same input; got {unique} unique'
        passed += 1; print(f'\u2705 Check 5: {unique} distinct prompt_fns registered')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add `--model` to each subcommand so users can choose the model per call
- Add a `--verbose` flag that prints the prompt before sending it to the LLM
- Add a `translate` subcommand: `--input` + `--lang` (target language),   prompt_fn builds `'Translate to {lang}: {input}'`
- Write `main.py` with `if __name__ == '__main__': sys.exit(main())` and a `pyproject.toml` — then install with `pip install -e .` and try `my-ai ask --input hello`
- Combine with Day 033 BatchProcessor: add a `batch` subcommand that reads   `--input` as a comma-separated list and processes all items concurrently